# Music Annotation Project V8.0f
## Complete Read-Only Perceptual-Impact Audit

**Runtime:** CPU. Choose **Runtime → Run all**.

V8.0f reconstructs the locked V8.0d output and a single counterfactual in which only the 49 V8.0e truth-UNCERTAIN final-PRINT coordinates are restored to frozen V7.8 `geometry_t80` values. It measures exact byte differences, connected visual regions, page-fit survival and local topology proxies, then creates blinded human-review material.

It does **not** train, rescore, recalibrate, alter thresholds or labels, change V7.8, use independent data, or authorise promotion. Automated topology proxies do not identify musical semantics; staff/notehead/glyph identity and normal-scale visibility require the blinded review form.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib pillow


## 1. Configuration, integrity gates and frozen inputs


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,platform,sys,html
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V8D_ROOT=ROOT/'observed_print_classifier_v8_0d'; V8E_ROOT=ROOT/'observed_print_classifier_v8_0e'; V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0f'
V8D_RUN_OVERRIDE=V8E_RUN_OVERRIDE=V75_RUN_OVERRIDE=None
EXPECTED_LOCK='66be107973e6b8ea8716e0d11e48116952e8e5fdf85fd2ea9292ce4b5f52ea61'
EXPECTED_DISPUTED=49; EXPECTED_FINAL_PRINT=124
DISPLAY_WIDTHS=[800,1200,1600]; PRIMARY_WIDTH=1200; GROUP_RADII=[0,1,2]; TOPOLOGY_THRESHOLDS=[160,190,220]
COMPONENT_MARGIN=48; TOPOLOGY_MARGIN=24; BLIND_SEED='v8.0f-fixed-blind-order-v1'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0f_read_only_perceptual_impact'; OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; AUDIT=OUT/'audit'; PAGES=OUT/'page_comparisons'; COMPONENTS=OUT/'component_comparisons'; BLIND=OUT/'blind_review'; GIFS=OUT/'blink_gifs'; IMAGES=OUT/'counterfactual_images'
for p in [OUT,TABLES,AUDIT,PAGES,COMPONENTS,BLIND,GIFS,IMAGES]: p.mkdir(parents=True,exist_ok=False)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_sha256(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def read_predictions(run):
    pq=run/'tables'/'all_eligible_oof_predictions.parquet'; cs=run/'tables'/'all_eligible_oof_predictions.csv'
    if pq.exists(): return pd.read_parquet(pq),pq
    if cs.exists(): return pd.read_csv(cs),cs
    raise RuntimeError('Missing locked V8.0d all-eligible prediction table.')

V8E=Path(V8E_RUN_OVERRIDE) if V8E_RUN_OVERRIDE else latest_complete(V8E_ROOT,['audit_manifest.json','uncertainty_audit_summary.json','tables/truth_uncertain_final_print_pixel_audit.csv'])
if V8E is None: raise RuntimeError('No complete V8.0e run found.')
em=json.loads((V8E/'audit_manifest.json').read_text()); es=json.loads((V8E/'uncertainty_audit_summary.json').read_text())
disputed_path=V8E/'tables'/'truth_uncertain_final_print_pixel_audit.csv'; disputed=pd.read_csv(disputed_path)
V8D=Path(V8D_RUN_OVERRIDE) if V8D_RUN_OVERRIDE else Path(em['source_v8d_run'])
if not V8D.exists(): raise RuntimeError(f'Locked V8.0d run absent: {V8D}')
dm=json.loads((V8D/'experiment_manifest.json').read_text()); dr=json.loads((V8D/'audit'/'replay_test.json').read_text()); predictions,prediction_path=read_predictions(V8D)
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('No complete V7.5 run found.')
required={'page','source_group','y','x','truth','decision','component_id','strict_class_r1','strict_class_r2'}
if required-set(disputed): raise RuntimeError(f'Missing V8.0e columns: {sorted(required-set(disputed))}')
checks={
 'v8d_manifest_lock':dm.get('run_lock_sha256')==EXPECTED_LOCK,
 'v8d_replay_lock':dr.get('run_lock_sha256')==EXPECTED_LOCK,
 'same_runtime_replay':dr.get('checks',{}).get('same_runtime_exact_replay_pass') is True,
 'previous_lock_found':dr.get('previous_completed_lock_found') is True,
 'previous_clean_match':dr.get('previous_clean_run_match') is True,
 'v8e_source_lock':em.get('source_v8d_run_lock_sha256')==EXPECTED_LOCK,
 'v8e_read_only':all(em.get(k) is False for k in ['model_trained','model_rescored','calibration_changed','threshold_changed','registered_labels_changed','v78_changed','independent_data_used']),
 'disputed_count':len(disputed)==EXPECTED_DISPUTED==es.get('truth_uncertain_final_print'),
 'truth_uncertain':bool(disputed.truth.eq(2).all()),'decision_print':bool(disputed.decision.eq('PRINT').all()),
 'mixed_r1':bool(disputed.strict_class_r1.eq('MIXED_UNRESOLVED').all()),'mixed_r2':bool(disputed.strict_class_r2.eq('MIXED_UNRESOLVED').all()),
 'unique_coordinates':not disputed.duplicated(['page','y','x']).any()}
(AUDIT/'input_lock_checks.json').write_text(json.dumps(checks,indent=2))
if not all(checks.values()): raise RuntimeError(f'Input lock gate failed: {checks}')
print('V8.0d',V8D); print('V8.0e',V8E); print('V7.5',V75); print('Output',OUT); print(json.dumps(checks,indent=2))


## 2. Reconstruct V8.0d, create the registered counterfactual and prove immutability


In [ ]:
status=pd.read_csv(V75/'tables'/'cache_status.csv'); cache_map={str(r.page):Path(r.cache) for _,r in status.iterrows()}
observed_dir=ROOT/'aligned'; observed_map={p.stem:p for p in observed_dir.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
affected_pages=sorted(disputed.page.unique()); selected=predictions[predictions.decision.eq('PRINT')].copy()
if len(selected)!=EXPECTED_FINAL_PRINT: raise RuntimeError(f'Expected {EXPECTED_FINAL_PRINT} final PRINT rows; found {len(selected)}')
assets={}; asset_rows=[]; coord_rows=[]; page_rows=[]
for page in affected_pages:
    if page not in observed_map or page not in cache_map: raise RuntimeError(f'{page}: missing observed/cache input')
    obs=cv2.imread(str(observed_map[page]),0)
    with np.load(cache_map[page]) as z:
        if not {'geometry_t80','mask'}.issubset(z.files): raise RuntimeError(f'{page}: cache lacks geometry_t80/mask')
        base=z['geometry_t80']; mask=z['mask']>0
    if obs is None or obs.shape!=base.shape or obs.shape!=mask.shape: raise RuntimeError(f'{page}: shape failure')
    ref=base.copy(); allq=selected[selected.page.eq(page)]; ref[allq.y.astype(int),allq.x.astype(int)]=obs[allq.y.astype(int),allq.x.astype(int)]
    saved=V8D/'audit'/f'{page}_challenger.png'; saved_equal=None
    if saved.exists():
        si=cv2.imread(str(saved),0); saved_equal=bool(si is not None and np.array_equal(si,ref))
        if not saved_equal: raise RuntimeError(f'{page}: reconstructed reference differs from saved V8.0d challenger')
    q=disputed[disputed.page.eq(page)]; yy=q.y.astype(int).to_numpy(); xx=q.x.astype(int).to_numpy()
    cf=ref.copy(); cf[yy,xx]=base[yy,xx]; allowed=np.zeros(ref.shape,bool); allowed[yy,xx]=True; diff=ref!=cf
    if np.any(diff&~allowed) or not np.array_equal(ref[~allowed],cf[~allowed]): raise RuntimeError(f'{page}: counterfactual drift')
    assets[page]={'observed':obs,'baseline':base,'mask':mask,'reference':ref,'counterfactual':cf,'diff':diff}
    cv2.imwrite(str(IMAGES/f'{page}_v8d_reference.png'),ref); cv2.imwrite(str(IMAGES/f'{page}_49_reverted_counterfactual.png'),cf)
    delta=np.abs(ref.astype(np.int16)-cf.astype(np.int16))
    for _,r in q.iterrows():
        y,x=int(r.y),int(r.x); coord_rows.append({'page':page,'source_group':r.source_group,'y':y,'x':x,'component_id':int(r.component_id),
         'v8d_value':int(ref[y,x]),'v78_value':int(cf[y,x]),'absolute_delta':int(delta[y,x]),'effective_byte_change':bool(diff[y,x])})
    page_rows.append({'page':page,'source_group':q.source_group.iloc[0],'nominal_disputed_coordinates':len(q),'effective_changed_pixels':int(diff.sum()),
     'zero_delta_registered_coordinates':int(len(q)-diff.sum()),'total_absolute_intensity_delta':int(delta.sum()),'maximum_absolute_intensity_delta':int(delta.max()),
     'mean_delta_over_effective_pixels':float(delta[diff].mean()) if diff.any() else 0.0,'fraction_of_page_pixels_changed':float(diff.mean()),
     'non_disputed_bytes_equal':True,'reference_sha256':array_sha256(ref),'counterfactual_sha256':array_sha256(cf)})
    asset_rows.append({'page':page,'observed_sha256':sha256_file(observed_map[page]),'cache_sha256':sha256_file(cache_map[page]),
     'baseline_array_sha256':array_sha256(base),'reference_array_sha256':array_sha256(ref),'saved_v8d_challenger_found':saved.exists(),
     'saved_v8d_challenger_equal':saved_equal,'height':obs.shape[0],'width':obs.shape[1],'final_print_rows_on_page':len(allq)})
coordinate_effects=pd.DataFrame(coord_rows); page_summary=pd.DataFrame(page_rows)
coordinate_effects.to_csv(TABLES/'registered_coordinate_effects.csv',index=False); page_summary.to_csv(TABLES/'page_difference_summary.csv',index=False)
pd.DataFrame(asset_rows).to_csv(AUDIT/'asset_and_reference_integrity.csv',index=False)
print(page_summary)


## 3. Connected regions, display-scale survival and topology proxies


In [ ]:
def region_labels(diff,radius):
    work=diff.astype(np.uint8)
    if radius: work=cv2.dilate(work,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1)))
    return cv2.connectedComponentsWithStats(work,8)
def crop_bounds(shape,ys,xs,m):
    h,w=shape; return max(0,int(np.min(ys))-m),min(h,int(np.max(ys))+m+1),max(0,int(np.min(xs))-m),min(w,int(np.max(xs))+m+1)
def binary_topology(gray,t):
    fg=(gray<t).astype(np.uint8); n,_,_,_=cv2.connectedComponentsWithStats(fg,8)
    bg=(np.pad(fg,1)==0).astype(np.uint8); nb,lab,_,_=cv2.connectedComponentsWithStats(bg,8)
    border=set(np.unique(np.r_[lab[0,:],lab[-1,:],lab[:,0],lab[:,-1]])); holes=sum(i not in border for i in range(1,nb)); comps=max(0,n-1)
    return comps,holes,comps-holes,int(fg.sum())
region_rows=[]; map_rows=[]; display_rows=[]; resized={}; component_rows=[]; topology_rows=[]
for page in affected_pages:
    z=assets[page]; ref,cf,diff=z['reference'],z['counterfactual'],z['diff']
    for radius in GROUP_RADII:
        n,lab,stats,cent=region_labels(diff,radius)
        for rid in range(1,n):
            ys,xs=np.where((lab==rid)&diff)
            if len(ys): region_rows.append({'page':page,'grouping_radius':radius,'region_id':rid,'effective_pixels':len(ys),'x_min':int(xs.min()),'x_max':int(xs.max()),'y_min':int(ys.min()),'y_max':int(ys.max()),'bbox_width':int(xs.max()-xs.min()+1),'bbox_height':int(ys.max()-ys.min()+1),'absolute_delta_sum':int(np.abs(ref[ys,xs].astype(int)-cf[ys,xs].astype(int)).sum()),'maximum_delta':int(np.abs(ref[ys,xs].astype(int)-cf[ys,xs].astype(int)).max())})
        for _,r in coordinate_effects[coordinate_effects.page.eq(page)].iterrows(): map_rows.append({'page':page,'y':int(r.y),'x':int(r.x),'grouping_radius':radius,'region_id':int(lab[int(r.y),int(r.x)]) if r.effective_byte_change else 0})
    h,w=ref.shape; resized[page]={}
    for width in DISPLAY_WIDTHS:
        height=max(1,round(h*width/w)); rr=cv2.resize(ref,(width,height),interpolation=cv2.INTER_AREA); cc=cv2.resize(cf,(width,height),interpolation=cv2.INTER_AREA); dd=np.abs(rr.astype(np.int16)-cc.astype(np.int16)); ch=dd>0; resized[page][width]=(rr,cc,dd)
        display_rows.append({'page':page,'display_width':width,'display_height':height,'changed_display_pixels':int(ch.sum()),'maximum_display_delta':int(dd.max()),'display_mae_all_pixels':float(dd.mean()),'display_mean_delta_where_changed':float(dd[ch].mean()) if ch.any() else 0.0,'display_fraction_pixels_changed':float(ch.mean())})
for (page,cid),q in disputed.groupby(['page','component_id']):
    ref,cf=assets[page]['reference'],assets[page]['counterfactual']; yy=q.y.astype(int).to_numpy(); xx=q.x.astype(int).to_numpy(); y0,y1,x0,x1=crop_bounds(ref.shape,yy,xx,COMPONENT_MARGIN); rc,cc=ref[y0:y1,x0:x1],cf[y0:y1,x0:x1]; dd=np.abs(rc.astype(np.int16)-cc.astype(np.int16)); eff=dd>0
    component_rows.append({'page':page,'source_group':q.source_group.iloc[0],'authoritative_component_id':int(cid),'registered_disputed_pixels':len(q),'effective_changed_pixels':int(eff.sum()),'x_min':int(xx.min()),'x_max':int(xx.max()),'y_min':int(yy.min()),'y_max':int(yy.max()),'crop_x0':x0,'crop_y0':y0,'crop_x1':x1,'crop_y1':y1,'maximum_delta':int(dd.max()),'mean_delta_where_changed':float(dd[eff].mean()) if eff.any() else 0.0})
    ty0,ty1,tx0,tx1=crop_bounds(ref.shape,yy,xx,TOPOLOGY_MARGIN); tr,tc=ref[ty0:ty1,tx0:tx1],cf[ty0:ty1,tx0:tx1]
    for t in TOPOLOGY_THRESHOLDS:
        a,b=binary_topology(tr,t),binary_topology(tc,t); topology_rows.append({'page':page,'authoritative_component_id':int(cid),'threshold':t,'reference_foreground_components':a[0],'counterfactual_foreground_components':b[0],'reference_holes':a[1],'counterfactual_holes':b[1],'reference_euler':a[2],'counterfactual_euler':b[2],'reference_dark_pixels':a[3],'counterfactual_dark_pixels':b[3],'foreground_component_count_changed':a[0]!=b[0],'hole_count_changed':a[1]!=b[1],'euler_changed':a[2]!=b[2]})
regions=pd.DataFrame(region_rows); coordinate_regions=pd.DataFrame(map_rows); display_metrics=pd.DataFrame(display_rows); component_summary=pd.DataFrame(component_rows); topology=pd.DataFrame(topology_rows)
regions.to_csv(TABLES/'connected_difference_regions.csv',index=False); coordinate_regions.to_csv(TABLES/'coordinate_to_region_map.csv',index=False); display_metrics.to_csv(TABLES/'display_scale_survival.csv',index=False); component_summary.to_csv(TABLES/'component_difference_summary.csv',index=False); topology.to_csv(TABLES/'threshold_topology_proxies.csv',index=False)
region_counts=regions.groupby(['page','grouping_radius']).size().unstack(fill_value=0); region_counts.to_csv(TABLES/'connected_region_counts_by_page_and_radius.csv')
print(region_counts); print(display_metrics); print(component_summary); print(topology)


## 4. Labelled and blinded visual review material


In [ ]:
def save_triptych(path,a,b,c,titles,title):
    fig,axes=plt.subplots(1,3,figsize=(18,8))
    for ax,img,ttl in zip(axes,[a,b,c],titles): ax.imshow(img,cmap='gray' if img.ndim==2 else None,vmin=0 if img.ndim==2 else None,vmax=255 if img.ndim==2 else None); ax.set_title(ttl); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); fig.savefig(path,dpi=150,bbox_inches='tight'); plt.close(fig)
def overlay(base,diff):
    rgb=cv2.cvtColor(base,cv2.COLOR_GRAY2RGB); rgb[diff>0]=[255,0,0]; return rgb
def is_ref_a(key): return hashlib.sha256((BLIND_SEED+'|'+key).encode()).hexdigest()[0]<'8'
blind_key=[]
for page in affected_pages:
    rr,cc,dd=resized[page][PRIMARY_WIDTH]; save_triptych(PAGES/f'{page}_pagefit_{PRIMARY_WIDTH}_comparison.png',rr,cc,overlay(cc,dd),['V8.0d reference','49-pixel-reverted counterfactual','Differences in red'],f'{page}: page-fit comparison')
    Image.fromarray(rr).save(GIFS/f'{page}_pagefit_{PRIMARY_WIDTH}_blink.gif',save_all=True,append_images=[Image.fromarray(cc)],duration=800,loop=0)
    ra=is_ref_a('page|'+page); a,b=(rr,cc) if ra else (cc,rr); fig,ax=plt.subplots(1,2,figsize=(14,9))
    for z,img,ttl in zip(ax,[a,b],['Image A','Image B']): z.imshow(img,cmap='gray',vmin=0,vmax=255); z.set_title(ttl); z.axis('off')
    fig.suptitle(f'{page} — blinded page-fit comparison'); plt.tight_layout(); fig.savefig(BLIND/f'{page}_pagefit_blind_AB.png',dpi=160,bbox_inches='tight'); plt.close(fig); blind_key.append({'review_unit_id':f'PAGE_{page}','reference_side':'A' if ra else 'B'})
for _,r in component_summary.iterrows():
    page,cid=r.page,int(r.authoritative_component_id); ref,cf=assets[page]['reference'],assets[page]['counterfactual']; y0,y1,x0,x1=map(int,[r.crop_y0,r.crop_y1,r.crop_x0,r.crop_x1]); rc,cc=ref[y0:y1,x0:x1],cf[y0:y1,x0:x1]; amp=np.clip(np.abs(rc.astype(np.int16)-cc.astype(np.int16))*8,0,255).astype(np.uint8)
    save_triptych(COMPONENTS/f'{page}_component_{cid}_original.png',rc,cc,amp,['V8.0d reference','Counterfactual','Absolute difference ×8'],f'{page} component {cid}')
    save_triptych(COMPONENTS/f'{page}_component_{cid}_4x.png',cv2.resize(rc,None,fx=4,fy=4,interpolation=cv2.INTER_NEAREST),cv2.resize(cc,None,fx=4,fy=4,interpolation=cv2.INTER_NEAREST),cv2.resize(amp,None,fx=4,fy=4,interpolation=cv2.INTER_NEAREST),['Reference 4×','Counterfactual 4×','Difference ×8, 4×'],f'{page} component {cid}')
    ra=is_ref_a(f'component|{page}|{cid}'); a,b=(rc,cc) if ra else (cc,rc); fig,ax=plt.subplots(1,2,figsize=(12,6))
    for z,img,ttl in zip(ax,[a,b],['Image A','Image B']): z.imshow(img,cmap='gray',vmin=0,vmax=255); z.set_title(ttl); z.axis('off')
    fig.suptitle(f'{page} component {cid} — blinded original scale'); plt.tight_layout(); fig.savefig(BLIND/f'{page}_component_{cid}_blind_AB.png',dpi=180,bbox_inches='tight'); plt.close(fig); blind_key.append({'review_unit_id':f'COMP_{page}_{cid}','reference_side':'A' if ra else 'B'})
(AUDIT/'blind_key.json').write_text(json.dumps(blind_key,indent=2))
print('Labelled pages',len(list(PAGES.glob('*.png'))),'component images',len(list(COMPONENTS.glob('*.png'))),'blind images',len(list(BLIND.glob('*.png'))))


## 5. Human review queues and HTML index


In [ ]:
review=[]
for page in affected_pages:
    ps=page_summary[page_summary.page.eq(page)].iloc[0]; review.append({'review_unit_id':f'PAGE_{page}','level':'page','page':page,'authoritative_component_id':'','image_file':f'blind_review/{page}_pagefit_blind_AB.png','effective_changed_pixels':int(ps.effective_changed_pixels),'normal_scale_difference_visible_yes_no_unclear':'','which_image_looks_better_A_B_same':'','confidence_1_to_5':'','reviewer_comment':''})
for _,r in component_summary.iterrows():
    page,cid=r.page,int(r.authoritative_component_id); review.append({'review_unit_id':f'COMP_{page}_{cid}','level':'component','page':page,'authoritative_component_id':cid,'image_file':f'blind_review/{page}_component_{cid}_blind_AB.png','effective_changed_pixels':int(r.effective_changed_pixels),'normal_scale_difference_visible_yes_no_unclear':'','which_image_looks_better_A_B_same':'','confidence_1_to_5':'','reviewer_comment':'','semantic_object_type_staff_notehead_stem_accidental_dynamic_text_other_unclear':'','semantic_impact_none_minor_material_unclear':'','topology_appears_changed_yes_no_unclear':''})
review=pd.DataFrame(review); review.to_csv(TABLES/'perceptual_review_queue.csv',index=False); review[review.level.eq('component')].to_csv(TABLES/'symbol_impact_review.csv',index=False)
sections=[]
for _,r in review.iterrows(): sections.append('<section><h2>'+html.escape(r.review_unit_id)+'</h2><p>'+str(int(r.effective_changed_pixels))+' effective changed pixels</p><img src="'+html.escape(str(r.image_file))+'" style="max-width:100%;height:auto"></section>')
html_doc='<!doctype html><html><head><meta charset="utf-8"><title>V8.0f blind review</title><style>body{font-family:sans-serif;max-width:1400px;margin:auto}section{border-bottom:1px solid #ccc;padding:20px}</style></head><body><h1>V8.0f blinded perceptual review</h1><p>Complete tables/perceptual_review_queue.csv before opening audit/blind_key.json.</p>'+''.join(sections)+'</body></html>'
(OUT/'review_index.html').write_text(html_doc,encoding='utf-8'); print(review); print('Review index',OUT/'review_index.html')


## 6. Summary and content-addressed output manifest


In [ ]:
topchange=topology.groupby(['page','authoritative_component_id']).agg(component_change=('foreground_component_count_changed','any'),hole_change=('hole_count_changed','any'),euler_change=('euler_changed','any')).reset_index()
summary={'source_v8d_lock_sha256':EXPECTED_LOCK,'source_v8e_table_sha256':sha256_file(disputed_path),'registered_disputed_coordinates':int(len(disputed)),'effective_byte_changes':int(coordinate_effects.effective_byte_change.sum()),'zero_delta_registered_coordinates':int((~coordinate_effects.effective_byte_change).sum()),'affected_pages':int(page_summary.effective_changed_pixels.gt(0).sum()),'affected_authoritative_components':int(component_summary.effective_changed_pixels.gt(0).sum()),'connected_region_counts_by_grouping_radius':{str(r):int((regions.grouping_radius==r).sum()) for r in GROUP_RADII},'largest_exact_connected_region_pixels':int(regions[regions.grouping_radius.eq(0)].effective_pixels.max()) if len(regions) else 0,'components_with_any_threshold_component_count_change':int(topchange.component_change.sum()),'components_with_any_threshold_hole_count_change':int(topchange.hole_change.sum()),'components_with_any_threshold_euler_change':int(topchange.euler_change.sum()),'semantic_symbols_automatically_identified':False,'human_visibility_review_completed':False,'interpretation':'Objective audit plus uncompleted blinded review queue; no promotion or registered-result change.'}
(OUT/'v80f_summary.json').write_text(json.dumps(summary,indent=2))
generated=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='audit_manifest.json': generated.append({'path':str(p.relative_to(OUT)),'sha256':sha256_file(p),'bytes':p.stat().st_size})
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.0f-read-only-perceptual-impact-audit','source_v8d_run':str(V8D),'source_v8e_run':str(V8E),'source_v75_run':str(V75),'source_prediction_sha256':sha256_file(prediction_path),'input_lock_checks':checks,'counterfactual_contract':'Only the 49 V8.0e registered truth-UNCERTAIN final PRINT coordinates may be replaced by frozen V7.8 geometry_t80 bytes.','model_trained':False,'model_rescored':False,'calibration_changed':False,'threshold_changed':False,'registered_labels_changed':False,'v78_changed':False,'independent_data_used':False,'freeze_permitted':False,'display_widths':DISPLAY_WIDTHS,'region_grouping_radii':GROUP_RADII,'topology_thresholds':TOPOLOGY_THRESHOLDS,'generated_files':generated,'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'audit_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(summary,indent=2)); print('Complete',OUT)


## Files to share after the run

**Mandatory quantitative files**

```text
v80f_summary.json
audit_manifest.json
audit/input_lock_checks.json
audit/asset_and_reference_integrity.csv
tables/page_difference_summary.csv
tables/component_difference_summary.csv
tables/connected_difference_regions.csv
tables/connected_region_counts_by_page_and_radius.csv
tables/display_scale_survival.csv
tables/threshold_topology_proxies.csv
```

**Mandatory blinded human-review files**

```text
review_index.html
tables/perceptual_review_queue.csv
tables/symbol_impact_review.csv
blind_review/*.png
```

Complete the review CSVs before opening `audit/blind_key.json`. Supporting visuals are in `page_comparisons/`, `component_comparisons/` and `blink_gifs/`.
